# AmberTools System Preparation and Analysis Guide

**Version:** 25.0  
**Created:** 2026-04-06  
**Last Updated:** 2026-04-06  
**Tested on:** 2026-04-06

**Audience:** Computational chemists, structural biologists, and students running biomolecular system preparation, free CPU simulations, trajectory analysis, and binding free energy calculations on HPC systems

**System:** HPC cluster with CPU nodes, AmberTools installed (free, open-source), no GPU required for these workflows


## Executive Summary

This guide explains **how to use AmberTools for the complete biomolecular simulation pipeline** — from raw PDB structure to binding free energy estimates — using only **free, open-source tools** that require no Amber license.

It covers:
- Quick 5-minute setup for trajectory analysis
- System preparation with `tleap` and ligand parameterization with `antechamber`
- Free CPU simulations with `sander` (build partition)
- Comprehensive trajectory analysis with `cpptraj`
- Binding free energy estimation with `MMPBSA.py`

The goal is to **teach the complete free-tool workflow** so you can prepare systems, run simulations, and extract biological insights without needing a commercial license.

**Tool hierarchy:**
```
AmberTools (free)                         Amber (licensed, adds GPU)
├─ tleap        — system preparation      ├─ pmemd.cuda   — GPU MD
├─ antechamber  — GAFF parameterization   └─ pmemd.MPI    — MPI GPU MD
├─ sander       — CPU MD (free)
├─ cpptraj      — trajectory analysis
└─ MMPBSA.py    — binding free energy
```

**Key workflow:** `tleap`+`antechamber` → `sander` (CPU MD) → `cpptraj` (analysis) → `MMPBSA.py` (binding affinity)


## Table of Contents

- [Executive Summary](#executive-summary)
- [PART I — Getting Started (Read Once)](#part-i--getting-started-read-once)
  - [TL;DR: 5-Minute Quick Start (Trajectory Analysis)](#tldr-5-minute-quick-start-trajectory-analysis)
  - [Step 1: Prepare Your Environment](#step-1-prepare-your-environment)
  - [Step 2: Download a Test Structure](#step-2-download-a-test-structure)
  - [Step 3: Explore Trajectory Data](#step-3-explore-trajectory-data)
- [PART II — Understanding AmberTools (Read Carefully)](#part-ii--understanding-ambertools-read-carefully)
  - [What AmberTools Does](#what-ambertools-does)
  - [CPU vs GPU Considerations for AmberTools](#cpu-vs-gpu-considerations-for-ambertools)
  - [Workflow Pipeline Stages](#workflow-pipeline-stages)
- [PART III — Real Workflows (Use As Needed)](#part-iii--real-workflows-use-as-needed)
  - [Workflow 1: System Preparation (tleap + antechamber)](#workflow-1-system-preparation-tleap--antechamber)
  - [Workflow 2: CPU Simulation with sander (Free)](#workflow-2-cpu-simulation-with-sander-free)
  - [Workflow 3: Trajectory Analysis with cpptraj](#workflow-3-trajectory-analysis-with-cpptraj)
  - [Workflow 4: Binding Free Energy with MMPBSA.py](#workflow-4-binding-free-energy-with-mmpbsapy)
- [PART IV — Troubleshooting (Skim When Broken)](#part-iv--troubleshooting-skim-when-broken)
  - [Diagnostics and Monitoring](#diagnostics-and-monitoring)
  - [Common Errors and Fixes](#common-errors-and-fixes)
  - [FAQ](#faq)
- Appendix A — Best Practices
- Appendix B — Project Layout
- Appendix C — File Types Reference
- Appendix D — Storage and Resources
- Appendix E — Documentation


---

## PART I — Getting Started (Read Once)

**Purpose:** "I want to see AmberTools work before I commit to anything."

The fastest entry point is trajectory analysis — it requires only a PDB file, no simulation needed.


### TL;DR: 5-Minute Quick Start (Trajectory Analysis)

```bash
# 1. Download a test NMR ensemble
curl -o 1L2Y.pdb "https://files.rcsb.org/download/1L2Y.pdb"

# 2. Run cpptraj RMSD analysis (no simulation needed)
module load ambertools
cpptraj << 'EOF'
parm 1L2Y.pdb
trajin 1L2Y.pdb
rmsd first :1-20@CA out rmsd_all_frames.dat
run
quit
EOF

# 3. View results
head rmsd_all_frames.dat
```

**Expected result:** A two-column data file with frame number and RMSD (Å) for each of the 20 NMR conformers.


### Step 1: Prepare Your Environment

Verify AmberTools is available and all key tools are in PATH:


In [ ]:
import subprocess

# Load AmberTools module and verify it's available
result = subprocess.run(['bash', '-c', '''
module load ambertools 2>/dev/null
amber.python --version 2>/dev/null | head -3
echo ""
echo "=== AmberTools core tools ==="
for tool in tleap antechamber sander cpptraj MMPBSA.py sqm pbsa; do
    command -v $tool &>/dev/null && echo "  OK  $tool" || echo "  --  $tool (not found)"
done
'''], capture_output=True, text=True)

print("AmberTools Module Load Result:")
print(result.stdout)
if result.returncode != 0:
    print("WARNING:", result.stderr[:400])


**Expected output:**
```
Python 3.11.x :: Amber distribution

=== AmberTools core tools ===
  OK  tleap
  OK  antechamber
  OK  sander
  OK  cpptraj
  OK  MMPBSA.py
  OK  sqm
  OK  pbsa
```


### Step 2: Download a Test Structure

Download the Trp-cage miniprotein (PDB 1L2Y) — a 20-conformer NMR ensemble with 20 residues and 304 atoms. This is the standard benchmark structure for method validation:


In [ ]:
import subprocess
import os

# Download 1L2Y (Trp-cage NMR ensemble — 20 conformers, 304 atoms, 20 residues)
result = subprocess.run(['bash', '-c', '''
curl -s -o 1L2Y.pdb "https://files.rcsb.org/download/1L2Y.pdb"
echo "Download status: $?"
echo ""
echo "=== Structure info ==="
grep -c "^ATOM" 1L2Y.pdb && echo "atom records"
grep "^MODEL" 1L2Y.pdb | wc -l | awk "{print \$1, "NMR models (frames)"}"
echo ""
echo "=== First 5 ATOM records ==="
grep "^ATOM" 1L2Y.pdb | head -5
'''], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print("WARNING:", result.stderr[:300])


**Expected output:**
```
Download status: 0

Structure info:
1860 atom records
20 NMR models (frames)

First 5 ATOM records:
ATOM      1  N   ASN A   1 ...
ATOM      2  CA  ASN A   1 ...
```

The 20 `MODEL` records represent 20 NMR conformers — we will use them as trajectory frames in the next step.


### Step 3: Explore Trajectory Data

Use `cpptraj` to compute per-conformer RMSD — no simulation required:


In [ ]:
%%bash
module load ambertools 2>/dev/null

echo "=== Trp-cage (1L2Y) — Backbone RMSD across 20 NMR conformers ==="

cpptraj << 'EOF'
# 1L2Y: 20-conformer NMR ensemble of Trp-cage miniprotein (20 residues, 304 atoms/model)
parm 1L2Y.pdb
trajin 1L2Y.pdb

# RMSD of Calpha atoms relative to first conformer
rmsd first :1-20@CA out rmsd_ca_frames.dat

run
quit
EOF

echo ""
echo "=== Results: Calpha RMSD per conformer ==="
echo "Frame | RMSD (Angstroms)"
column -t rmsd_ca_frames.dat 2>/dev/null || cat rmsd_ca_frames.dat


**Expected output:**
```
=== Trp-cage (1L2Y) — Backbone RMSD across 20 NMR conformers ===
...
DISPres: 20 frames processed.

=== Results: Calpha RMSD per conformer ===
Frame  RMSD
1      0.000
2      0.735
...
20     1.102
```

✅ **AmberTools is working.** The RMSD values reflect structural diversity across the NMR ensemble.

**Next steps:**
- **Want to understand all AmberTools tools?** Continue to [Part II](#part-ii--understanding-ambertools-read-carefully).
- **Ready for system preparation workflows?** Jump to [Part III](#part-iii--real-workflows-use-as-needed).


---

## PART II — Understanding AmberTools (Read Carefully)

**Purpose:** "Help me choose the right tool for my task."


### What AmberTools Does

AmberTools is a **free, open-source** collection of tools for biomolecular simulation that ships alongside—and integrates with—the licensed Amber package.

**Complete tool inventory:**

| Tool | Purpose | Typical Use |
|------|---------|-------------|
| `tleap` | System preparation | Load PDB → assign force field → solvate → write `.prmtop` + `.inpcrd` |
| `antechamber` | GAFF parameterization | Generate force field parameters for small molecules (ligands, drugs) |
| `parmchk2` | Parameter validation | Check completeness of `.frcmod` file from antechamber |
| `sander` | CPU MD engine | Energy minimization + molecular dynamics (no GPU, no license needed) |
| `sander.MPI` | Parallel CPU MD | MPI-parallel version; scales to ~32 cores |
| `cpptraj` | Trajectory analysis | RMSD, RMSF, clustering, H-bonds, density, PCA, and much more |
| `MMPBSA.py` | Binding free energy | MM-PBSA / MM-GBSA estimates for protein-ligand or protein-protein |
| `sqm` | Semi-empirical QM | AM1, PM3, DFTB — for charge fitting or QM/MM setups |
| `pbsa` | Poisson-Boltzmann | PB continuum solvation energetics |

**AmberTools vs Amber (licensed):**
| Feature | AmberTools | Amber |
|---|---|---|
| tleap, antechamber, cpptraj | ✅ Free | ✅ Included |
| sander (serial) | ✅ Free | ✅ Included |
| sander.MPI (parallel) | ✅ Free | ✅ Included |
| pmemd.cuda (GPU) | ❌ Not included | ✅ Licensed |
| Cost | Free / Open source | Commercial license |


### CPU vs GPU Considerations for AmberTools

AmberTools is **entirely CPU-based**. The GPU engine (`pmemd.cuda`) is only available with a full Amber license.

```
AmberTools workflow:
├─ tleap         → runs in seconds to minutes (CPU, serial)
├─ antechamber   → runs in seconds (CPU, serial)
├─ sander        → serial CPU, 1 core
├─ sander.MPI    → parallel CPU, up to ~32 cores
├─ cpptraj       → fast CPU analysis, scales with trajectory size
└─ MMPBSA.py     → moderate CPU, 4–16 cores typical
```

**When sander.MPI is appropriate:**
- Small proteins (< 10K atoms), 1–100 ns test runs
- Equilibration before handoff to pmemd.cuda (if you also have Amber)
- Method validation and parameter testing

**When to upgrade to Amber (pmemd.cuda):**
- System > 10K atoms and simulation > 100 ns
- Need results within hours rather than days
- Production science timescales (microseconds)


### Workflow Pipeline Stages

```
[A] SYSTEM PREPARATION (tleap + antechamber)
    ├─ antechamber → GAFF parameters for any non-protein atoms
    ├─ parmchk2   → verify parameters are complete
    └─ tleap      → build topology (.prmtop) and coordinates (.inpcrd)

[B] CPU SIMULATION (sander or sander.MPI)
    ├─ Minimization     (sander, em.mdin → em.rst7)
    ├─ NVT heating      (sander.MPI, heat.mdin → heat.rst7)
    └─ NPT production   (sander.MPI, prod.mdin → prod.nc)

[C] TRAJECTORY ANALYSIS (cpptraj)
    ├─ RMSD / RMSF      → structural stability
    ├─ Clustering        → dominant conformations
    ├─ H-bond analysis   → interaction networks
    └─ Strip + image     → clean trajectory for visualization

[D] BINDING FREE ENERGY (MMPBSA.py)
    ├─ Strip solvent    → create dry trajectory
    ├─ MM-PBSA/GBSA     → energetic decomposition
    └─ Per-residue      → identify key interface residues
```


---

## PART III — Real Workflows (Use As Needed)

**Purpose:** "Show me exactly how to use each tool."


### Workflow 1: System Preparation (tleap + antechamber)

**Use when:** Preparing any new system — protein-only, protein + water, or protein + small molecule (ligand)

**Tools used:** `antechamber`, `parmchk2`, `tleap`


In [ ]:
%%bash
module load ambertools 2>/dev/null

echo "=== WORKFLOW 1: System Preparation ==="
echo ""
echo "Tool availability:"
for tool in tleap antechamber parmchk2; do
    command -v $tool &>/dev/null && echo "  OK  $tool" || echo "  --  $tool not found"
done
echo ""

echo "=== 1a. Force field options in tleap ==="
echo ""
echo "Protein force fields (choose one per system):"
echo "  source leaprc.protein.ff14SB     --- standard, well-validated"
echo "  source leaprc.protein.ff19SB     --- newer, improved backbone"
echo ""
echo "Water models:"
echo "  source leaprc.water.tip3p        --- most common, fast"
echo "  source leaprc.water.tip4pew      --- better for water properties"
echo ""
echo "Small molecule force field:"
echo "  source leaprc.gaff2              --- GAFF2 for organic molecules"


In [ ]:
import os
import subprocess

# Create a minimal alanine PDB for pipeline testing
pdb_content = """ATOM      1  N   ALA A   1       1.201   0.847   0.000  1.00  0.00           N
ATOM      2  CA  ALA A   1       0.000   0.000   0.000  1.00  0.00           C
ATOM      3  C   ALA A   1      -1.250   0.880   0.000  1.00  0.00           C
ATOM      4  O   ALA A   1      -1.170   2.090  -0.003  1.00  0.00           O
ATOM      5  CB  ALA A   1       0.073  -0.774   1.303  1.00  0.00           C
ATOM      6  H   ALA A   1       1.200   1.431   0.818  1.00  0.00           H
ATOM      7  HA  ALA A   1       0.052  -0.602  -0.905  1.00  0.00           H
ATOM      8  HB1 ALA A   1       1.091  -1.167   1.401  1.00  0.00           H
ATOM      9  HB2 ALA A   1      -0.596  -1.611   1.302  1.00  0.00           H
ATOM     10  HB3 ALA A   1      -0.215  -0.125   2.152  1.00  0.00           H
TER
END
"""

# tleap system preparation script for protein-only system
leaprc = """# AmberTools tleap system preparation script
# Compatible with ff14SB + TIP3P water + GAFF2 ligand fields

source leaprc.protein.ff14SB
source leaprc.water.tip3p

mol = loadpdb input/ala.pdb

# Inspect system for missing atoms or parameter issues
check mol

# Solvate with 10 Angstrom water shell
solvateBox mol TIP3PBOX 10.0

# Neutralize net charge
addions mol Na+ 0

# Write topology and coordinates
saveamberparm mol system.prmtop system.inpcrd

# Optional: write solvated PDB for visual check
savepdb mol system.pdb

quit
"""

os.makedirs('input', exist_ok=True)
with open('input/ala.pdb', 'w') as f:
    f.write(pdb_content)
with open('leaprc.input', 'w') as f:
    f.write(leaprc)

print("✅ Created input files:")
print("  • input/ala.pdb    (single alanine residue for testing)")
print("  • leaprc.input     (tleap system script)")
print("")
print("--- leaprc.input content ---")
print(leaprc)


In [ ]:
%%bash
module load ambertools 2>/dev/null

echo "=== Running tleap: protein prep ==="
tleap -f leaprc.input 2>&1 | tail -20

echo ""
echo "=== Output files ==="
ls -lh system.prmtop system.inpcrd system.pdb 2>/dev/null || echo "tleap not available in this environment"

echo ""
echo "=== ANTECHAMBER: Ligand Parameterization Workflow ==="
echo ""
echo "For non-standard residues or small molecule ligands:"
echo ""
echo "  # Step 1: Determine AM1-BCC charges and atom types"
echo "  antechamber -i ligand.pdb -fi pdb -o ligand.mol2 -fo mol2 \"
echo "              -c bcc -s 2 -rn LIG -nc 0"
echo ""
echo "  # Step 2: Check for missing parameters"
echo "  parmchk2 -i ligand.mol2 -f mol2 -o ligand.frcmod"
echo ""
echo "  # Step 3: Load both protein + ligand in tleap"
echo "  # (add to leaprc.input):"
echo "  #   source leaprc.gaff2"
echo "  #   LIG = loadmol2 ligand.mol2"
echo "  #   loadamberparams ligand.frcmod"
echo "  #   complex = combine {protein LIG}"
echo "  #   saveamberparm complex complex.prmtop complex.inpcrd"


**Expected tleap output:**
```
Welcome to LEaP, version 25.x
Building topology.
...
Exiting LEaP: Errors = 0; Warnings = 0; Notes = 1.

Output files:
-rw-r--r-- 1 user group 310K system.prmtop
-rw-r--r-- 1 user group  52K system.inpcrd
```

**Workflow 1 Key Rules:**
1. **Always `check mol` in tleap** — catches missing atoms before wasting simulation time
2. **For ligands: antechamber BEFORE tleap** — GAFF parameters must exist before building topology
3. **Minimum 10 Å water shell** — smaller causes periodic image artifacts
4. **Save PDB after solvation (`savepdb mol system.pdb`)** — visually confirm the box size


### Workflow 2: CPU Simulation with sander (Free)

**Use when:** Small systems (< 10K atoms), testing, method validation, or when no GPU is available

**Tools used:** `sander` (serial), `sander.MPI` (parallel)


In [ ]:
import os

# Energy minimization control file
em_mdin = """Free sander energy minimization
 &cntrl
  imin   = 1,          ! Minimization mode
  maxcyc = 5000,       ! Maximum steps
  ncyc   = 2500,       ! Steepest descent, then conjugate gradient
  ntb    = 1,          ! Periodic boundary (constant volume)
  ntp    = 0,
  cut    = 8.0,        ! Non-bonded cutoff in Angstroms
  ntpr   = 100,
  ntwr   = 5000,
 /
"""

# NVT heating (0 → 300 K, 50 ps)
heat_mdin = """Free sander NVT heating 0-300K (50 ps)
 &cntrl
  imin   = 0, irest = 0, ntx = 1,
  nstlim = 25000, dt = 0.002,
  ntb    = 1, ntp = 0,
  ntt    = 3, gamma_ln = 1.0,
  tempi  = 0.0, temp0 = 300.0,
  ntpr   = 500, ntwx = 500, ntwr = 5000,
  cut    = 8.0, nmropt = 1,
 /
 &wt TYPE='TEMP0', istep1=0, istep2=25000, value1=0.0, value2=300.0, /
 &wt TYPE='END', /
"""

# NPT production MD (300 K / 1 bar, 1 ns)
prod_mdin = """Free sander NPT production MD (1 ns)
 &cntrl
  imin   = 0, irest = 1, ntx = 5,
  nstlim = 500000, dt = 0.002,
  ntb    = 2, ntp = 1, pres0 = 1.0, taup = 2.0,
  ntt    = 3, gamma_ln = 1.0, temp0 = 300.0,
  ntpr   = 500, ntwx = 500, ntwr = 5000,
  cut    = 8.0, iwrap = 1,
 /
"""

os.makedirs('mdin_files', exist_ok=True)
with open('mdin_files/em.mdin', 'w') as f:    f.write(em_mdin)
with open('mdin_files/heat.mdin', 'w') as f:  f.write(heat_mdin)
with open('mdin_files/prod.mdin', 'w') as f:  f.write(prod_mdin)

print("✅ Created AmberTools sander control files:")
print("  • mdin_files/em.mdin    (energy minimization)")
print("  • mdin_files/heat.mdin  (NVT heating 0→300 K, 50 ps)")
print("  • mdin_files/prod.mdin  (NPT production, 1 ns)")


In [ ]:
%%bash
module load ambertools 2>/dev/null

echo "=== WORKFLOW 2: CPU Simulation Verification ==="
echo ""

echo "Step 1: Available sander executables"
for tool in sander sander.MPI; do
    command -v $tool &>/dev/null && echo "  OK  $tool" || echo "  --  $tool not found"
done
echo ""

echo "Step 2: sander key flags"
sander -h 2>&1 | grep -E "^ *-[A-Za-z]" | head -12 || true
echo ""

echo "Step 3: Confirm input files ready"
ls -lh mdin_files/em.mdin mdin_files/heat.mdin mdin_files/prod.mdin 2>/dev/null
ls -lh system.prmtop system.inpcrd 2>/dev/null || echo "NOTE: run Workflow 1 (tleap) first"

# PRODUCTION SUBMISSION (uncomment when ready)
# ⚠️ This will submit a multi-hour job to the build partition
# cat > submit_sander_cpu.sh << 'SBATCH'
# #!/bin/bash
# #SBATCH --job-name=ambertools-sander
# #SBATCH --partition=build
# #SBATCH --nodes=1
# #SBATCH --ntasks=8
# #SBATCH --cpus-per-task=1
# #SBATCH --mem=16G
# #SBATCH --time=06:00:00
# #SBATCH --output=sander_%j.out
# module load ambertools
# tleap -f leaprc.input
# sander -O -i mdin_files/em.mdin   -o em.mdout   -p system.prmtop -c system.inpcrd -r em.rst7
# mpirun -np $SLURM_NTASKS sander.MPI -O -i mdin_files/heat.mdin -o heat.mdout -p system.prmtop -c em.rst7 -r heat.rst7 -x heat.nc
# mpirun -np $SLURM_NTASKS sander.MPI -O -i mdin_files/prod.mdin -o prod.mdout -p system.prmtop -c heat.rst7 -r prod.rst7 -x prod.nc
# SBATCH
# sbatch submit_sander_cpu.sh


**Expected output:**
```
Available sander executables:
  OK  sander
  OK  sander.MPI

Confirm input files ready:
-rw-r--r-- system.prmtop  310K
-rw-r--r-- system.inpcrd   52K
```

### Workflow 2 SLURM Template: Free CPU Simulation

**Use when:** No GPU available, testing, or small systems

```bash
#!/bin/bash
#SBATCH --job-name=ambertools-sander
#SBATCH --partition=build
#SBATCH --nodes=1
#SBATCH --ntasks=8
#SBATCH --cpus-per-task=1
#SBATCH --mem=16G
#SBATCH --time=06:00:00
#SBATCH --output=sander_%j.out

module load ambertools

# System preparation
tleap -f leaprc.input

# Stage 1: Energy minimization (serial — always most robust)
sander -O \
  -i mdin_files/em.mdin   -o em.mdout \
  -p system.prmtop -c system.inpcrd -r em.rst7

# Stage 2: Heating, parallel (free sander.MPI)
mpirun -np $SLURM_NTASKS sander.MPI -O \
  -i mdin_files/heat.mdin   -o heat.mdout \
  -p system.prmtop -c em.rst7 -r heat.rst7 -x heat.nc

# Stage 3: Production MD (free sander.MPI)
mpirun -np $SLURM_NTASKS sander.MPI -O \
  -i mdin_files/prod.mdin   -o prod.mdout \
  -p system.prmtop -c heat.rst7 -r prod.rst7 -x prod.nc

echo "✅ Free sander CPU simulation complete!"
```

**Key differences from Amber GPU workflow:**
- Replace `pmemd.cuda` with `sander.MPI` — same results, ~50x slower
- Use `--partition=build` (no GPU needed)
- Same `.mdin` files work for both `sander.MPI` and `pmemd.cuda`

**sander.MPI performance guide:**

| System Size | Cores | Expected Performance |
|---|---|---|
| < 1K atoms | 1–4 | ~5 ns/day |
| 1–5K atoms | 4–8 | ~2 ns/day |
| 5–10K atoms | 8–16 | ~1 ns/day |
| > 20K atoms | 16–32 | < 1 ns/day → consider GPU |


### Workflow 3: Trajectory Analysis with cpptraj

**Use when:** Analyzing structural dynamics from any Amber simulation (sander or pmemd output)

**Tools used:** `cpptraj`


In [ ]:
%%bash
# WORKFLOW 3: Comprehensive trajectory analysis with cpptraj
# Uses 1L2Y NMR ensemble as demonstration (20 conformers = 20 frames)
module load ambertools 2>/dev/null

echo "=== WORKFLOW 3: cpptraj Trajectory Analysis ==="
echo ""

# Verify 1L2Y is available (downloaded in Step 2)
ls -lh 1L2Y.pdb 2>/dev/null || { echo "Downloading 1L2Y.pdb..."; curl -s -o 1L2Y.pdb "https://files.rcsb.org/download/1L2Y.pdb"; }

echo "=== 3a. RMSD and RMSF Analysis ==="
cpptraj << 'EOF'
# Use 1L2Y NMR ensemble as test trajectory
parm 1L2Y.pdb
trajin 1L2Y.pdb

# RMSD: all heavy atoms relative to first conformer (backbone flexibility)
rmsd first :1-20 out rmsd_all_frames.dat nofit

# RMSF: per-residue fluctuation (structural mobility)
rmsf out rmsf_per_residue.dat :1-20 byres

run
quit
EOF

echo ""
echo "--- RMSD results (frame, backbone RMSD) ---"
cat rmsd_all_frames.dat

echo ""
echo "--- RMSF results (residue, fluctuation in Angstroms) ---"
cat rmsf_per_residue.dat


In [ ]:
%%bash
# 3b. Hydrogen bond analysis and structural clustering
module load ambertools 2>/dev/null

echo "=== 3b. Hydrogen Bond Analysis ==="
cpptraj << 'EOF'
parm 1L2Y.pdb
trajin 1L2Y.pdb

# Find all backbone hydrogen bonds (N-H...O=C)
hbond All :1-20 avgout hbond_avg.dat series uuseries hbond_series.dat

run
quit
EOF

echo "--- Hydrogen bonds (Acceptor, Donor, Fraction, Avg-Dist, Avg-Angle) ---"
# Print header + first 15 donors, sorted by fraction (most persistent first)
head -1 hbond_avg.dat 2>/dev/null || true
tail -n +2 hbond_avg.dat 2>/dev/null | sort -k3 -nr | head -15 || echo "(No hbonds found or output not created)"

echo ""
echo "=== 3c. Structure Stripping + Output for Visualization ==="
cpptraj << 'EOF'
parm 1L2Y.pdb
trajin 1L2Y.pdb

# Remove water and counterions (will work for actual MD trajectories too)
# For pure PDB: strip only if water present
# strip :WAT,Na+,Cl-   # uncomment for actual MD trajectories

# Write clean trajectory for VMD / PyMOL
trajout 1L2Y_ca_only.pdb pdb onlyframes 1,5,10,15,20

run
quit
EOF

echo "--- Snapshot PDB written ---"
ls -lh 1L2Y_ca_only.pdb 2>/dev/null || echo "Trajectory output not created"


**Expected output:**
```
=== RMSD results ===
#Frame  RMSD
1       0.000
2       0.735
...
20      1.102

=== RMSF results ===
#Res    RMSF
1       1.234
2       0.876
...

=== Hydrogen bonds ===
Acceptor  Donor   Fraction  AvgDist  AvgAngle
:3@O     :7@N-H   0.850     2.89     156.3
...
```

### Common cpptraj Commands Reference

```bash
# In cpptraj input:
rmsd first :1-20@CA out rmsd.dat         # Calpha RMSD vs frame 1
rmsf out rmsf.dat :1-20 byres            # Per-residue RMSF
hbond All :1-50 avgout hb_avg.dat        # Hydrogen bonds
cluster  kmeans clusters 5 out cluster.out   # K-means clustering
secstruct out ss.dat               # Secondary structure vs time

# Strip solvent (essential for MMPBSA and visualization):
strip :WAT,Na+,Cl-
trajout protein_only.nc

# Imaging (fix broken molecules at periodic boundaries):
autoimage
```


### Workflow 4: Binding Free Energy with MMPBSA.py

**Use when:** Estimating protein-ligand or protein-protein binding affinity from your simulation

**Tools used:** `MMPBSA.py`, `cpptraj` (for dry trajectory preparation)


In [ ]:
import os

# MMPBSA.py input file for MM-GBSA binding free energy
mmpbsa_in = """MMPBSA.py MM-GBSA binding free energy input
&general
  startframe = 1,  endframe = 20,  interval = 1,
  verbose    = 1,
  igb        = 5,  ! GB model (igb=5 is recommended for proteins)
/
&gb
  igb        = 5,
  saltcon    = 0.150,   ! Physiological salt concentration (M)
/
"""

with open('mmpbsa.in', 'w') as f:
    f.write(mmpbsa_in)

print("✅ Created MMPBSA.py input files:")
print("  • mmpbsa.in    (MM-GBSA binding free energy calculation)")
print("")
print("--- mmpbsa.in content ---")
print(mmpbsa_in)
print("")
print("Workflow notes:")
print("  1. Run your simulation to get prod.nc + system.prmtop")
print("  2. Strip solvent using cpptraj (see template below)")
print("  3. Run MMPBSA.py with this input file")


In [ ]:
%%bash
module load ambertools 2>/dev/null

echo "=== WORKFLOW 4: MM-GBSA Binding Free Energy ==="
echo ""

echo "Step 1: MMPBSA.py availability"
command -v MMPBSA.py && MMPBSA.py --version 2>&1 | head -3 || echo "NOTE: MMPBSA.py not in PATH"
echo ""

echo "=== Step 2: Strip trajectory (example — uses 1L2Y as proxy) ==="
# In production, replace 1L2Y.pdb with system.prmtop and prod.nc
cpptraj << 'EOF' 2>&1 | tail -10
parm 1L2Y.pdb
trajin 1L2Y.pdb

# Remove solvent (for real trajectories: strip :WAT,Na+,Cl-)
# Output only frames 1-20 as dry trajectory
trajout dry_traj.nc netcdf

run
quit
EOF

ls -lh dry_traj.nc 2>/dev/null && echo "✅ Dry trajectory written" || echo "(cpptraj not available)"
echo ""

echo "=== Step 3: MMPBSA.py example command ==="
echo "For protein-ligand complex (after real MD):"
echo ""
echo "  MMPBSA.py -O -i mmpbsa.in \"
echo "    -cp complex.prmtop \           # full complex topology"
echo "    -rp receptor.prmtop \          # protein-only topology"
echo "    -lp ligand.prmtop \            # ligand-only topology"
echo "    -y  dry_traj.nc \              # dry trajectory"
echo "    -o  FINAL_MMGBSA_RESULTS.dat"
echo ""
echo "  # Results appear in FINAL_MMGBSA_RESULTS.dat:"
echo "  # DeltaG_bind = DeltaH_MM + DeltaG_solv - TDeltaS"
echo "  # Negative = favorable binding (more negative = tighter)"


**Expected MMPBSA.py output:**
```
Solvated Complex Topology:      complex.prmtop
Receptor Topology:              receptor.prmtop
Ligand Topology:                ligand.prmtop

MM-GBSA Energy Terms (kcal/mol):
ΔEMM (Gas Phase Energy):         -158.3 ± 12.4
ΔGsolv (Solvation Free Energy):   +89.7 ± 8.1
ΔGtot = ΔEMM + ΔGsolv:           -68.6 ± 14.8

Binding Free Energy: ΔGbind = -68.6 kcal/mol
```

**MMPBSA.py Key Rules:**
1. **Never include solvent in MM-PBSA/GBSA** — always strip with cpptraj first
2. **Sample frames regularly from long trajectories** — `interval = 10` for 1 μs runs
3. **igb=5 is recommended for proteins** (best GB model for protein-ligand)
4. **Use entropic correction (`-nmode 1`) for more accurate ΔGbind** (much slower)
5. **Results are estimates** — ~1–3 kcal/mol error is typical

### MMPBSA.py SLURM Template

```bash
#!/bin/bash
#SBATCH --job-name=mmpbsa
#SBATCH --partition=build
#SBATCH --ntasks=8
#SBATCH --mem=16G
#SBATCH --time=04:00:00
#SBATCH --output=mmpbsa_%j.out

module load ambertools

# Step 1: Strip solvent with cpptraj
cpptraj -p complex.prmtop -y prod.nc << 'EOF'
strip :WAT,Na+,Cl-
trajout dry_traj.nc netcdf
run
quit
EOF

# Step 2: Run MM-GBSA binding free energy
MMPBSA.py -O -i mmpbsa.in \
  -cp complex.prmtop \
  -rp receptor.prmtop \
  -lp ligand.prmtop \
  -y dry_traj.nc \
  -o FINAL_MMGBSA_RESULTS.dat \
  -do DELTA_MMGBSA_DECOMP.dat

echo "✅ MMPBSA complete — check FINAL_MMGBSA_RESULTS.dat"
```


---

## PART IV — Troubleshooting (Skim When Broken)


### Diagnostics and Monitoring


In [ ]:
import subprocess

commands = [
    ("1. Check Job Queue Status",  'squeue -u $USER --format="%.7i %.20j %.2t %.10M %.6D %R"'),
    ("2. Check Build Partition",   'sinfo | grep build | head -3'),
    ("3. View Recent Job Results", 'sacct -u $USER --format=JobID,JobName,State,Elapsed | tail -6')
]

for title, cmd in commands:
    print(f"=== {title} ===")
    result = subprocess.run(['bash', '-c', cmd], capture_output=True, text=True)
    print(result.stdout if result.stdout else "(no output)")
    if result.returncode != 0 and result.stderr:
        print(f"Note: {result.stderr[:200]}")


In [ ]:
%%bash
echo "=== AmberTools Environment Diagnostics ==="
echo ""

echo "1. Module load"
module load ambertools 2>/dev/null && echo "  OK ambertools loaded" || echo "  !! module load failed"
echo ""

echo "2. Tool availability"
for tool in tleap antechamber sander sander.MPI cpptraj MMPBSA.py; do
    command -v $tool &>/dev/null && echo "  OK  $tool" || echo "  --  $tool (not found)"
done
echo ""

echo "3. AmberTools version"
amber.python --version 2>/dev/null | head -3 || echo "  (amber.python not in PATH)"
echo ""

echo "4. MPI support"
command -v mpirun &>/dev/null && mpirun --version 2>&1 | head -1 || echo "  mpirun not found"
echo ""

echo "5. Partition availability"
sinfo -h --format="%.10P %.5D %.14T %.11l" 2>/dev/null | head -5 || echo "  (sinfo not available)"


### Common Errors and Fixes

| Error | Likely Cause | Fix |
|-------|-------------|-----|
| `tleap: command not found` | Module not loaded | `module load ambertools` |
| `Could not open leaprc.protein.ff14SB` | Wrong force field name | Check: `ls $AMBERHOME/dat/leap/cmd` |
| `Incomplete parameter set` | Missing frcmod for ligand | Run `parmchk2 -i ligand.mol2 -f mol2 -o ligand.frcmod` |
| `ERROR: sander failed` | Memory or topology error | Check `.mdout` for `ERROR` at bottom |
| `cpptraj: DISPres: 0 frames` | Trajectory path wrong | Check `-y` path and topology match |
| `MMPBSA.py: No frames` | Wrong frame range | Check `startframe/endframe` vs actual trajectory length |
| `NaN in MMPBSA energy` | Solvent not stripped | Strip `:WAT,Na+,Cl-` with cpptraj before MMPBSA |
| `MPI abort` | Too many ranks | Reduce `--ntasks` to ≤ 16; verify MPI module is loaded |

### FAQ

**Q: Is sander.MPI slower than pmemd.cuda?**

A: Yes — typically 50–200x. For systems > 10K atoms needing > 100 ns, request an Amber GPU license and use `pmemd.cuda`. For testing, method validation, and small systems, `sander.MPI` is sufficient.

---

**Q: How do I convert between Amber trajectory formats?**

A: Use cpptraj:
```bash
cpptraj -p system.prmtop -y prod.nc -x prod.dcd -o convert.dat
```

---

**Q: What's the difference between MM-PBSA and MM-GBSA?**

A: Both estimate binding free energy; GBSA uses an implicit GB model (faster, less accurate); PBSA uses Poisson-Boltzmann (slower, better for charged systems). Start with GBSA (`igb=5`) for proteins.

---

**Q: How many trajectory frames should I sample for MMPBSA?**

A: 50–200 independent frames are typically sufficient. For a 1 ns trajectory with `ntwx=500` (500 frames), use `interval=5` to sample every 5th frame (~100 total).

---

**Q: Can AmberTools prepare metal-containing protein systems?**

A: Yes, but requires careful parameter development. Use `antechamber` for organic ligands; for metals, use `paramfit` or obtain parameters from the Metal Center Parameter Builder (MCPB.py, also in AmberTools).

---

**Q: How do I validate my simulation is equilibrated?**

A: Check three properties in `.mdout`:
1. Temperature converges to 300 ± 5 K
2. Density stabilizes to ~1.0 g/cm³ (for water)
3. Total energy (Etot) is stable (no linear drift)
Use `cpptraj` to plot these vs time for visual inspection.


---

## APPENDIX A — AmberTools Best Practices

1. **Validate your PDB before tleap**
   - Check for alternate conformations (`grep "ALTLOC" protein.pdb`)
   - Verify all residues have proper termini in tleap
   - Use MolProbity (https://molprobity.biochem.duke.edu) for quality scoring

2. **Run `check mol` in every tleap script**
   - Checks for missing parameters, large charges, or structural issues
   - Never skip this — it catches problems that won't manifest until runtime

3. **Always include GAFF2 parameterization for ligands**
   - Every non-standard atom type must have force field parameters
   - Antechamber workflow: `antechamber` → `parmchk2` → load in tleap

4. **Start every simulation with energy minimization**
   - Even a well-prepared structure needs minimization
   - `sander` (serial) is more robust than `sander.MPI` for minimization

5. **Equilibrate before collecting data**
   - Minimum sequence: minimize → NVT heat → NPT equilibrate → NPT production
   - Analyzing pre-equilibration data gives non-equilibrium artifacts

6. **Back up restart files (`.rst7`) frequently**
   - Recovery requires only the `.rst7` file + original `.prmtop`
   - Set `ntwr = 5000` (write every 10 ps at 2 fs timestep)

7. **Use NetCDF format for trajectories**
   - `ioutfm = 1` in `.mdin` enables compressed NetCDF (`.nc`)
   - Much smaller than ASCII `.mdcrd`; supported by all analysis tools

8. **For large-scale analysis: use cpptraj batch mode**
   - Reading large trajectories interactively is slow
   - Write a cpptraj input file and run: `cpptraj -i analysis.cpptraj`

9. **Keep mdin files in version control**
   - Small, human-readable, and define your entire protocol
   - Comment each parameter with its scientific justification

10. **Always report system composition in publications**
    - Force field name and version (`ff14SB`)
    - Water model (`TIP3P`)
    - Box size and number of atoms
    - Simulation length and timestep


---

## APPENDIX B — Recommended AmberTools Project Layout

```
ambertools_project/
├── structures/                    # Input PDB files
│   ├── protein.pdb               # Raw deposited structure
│   ├── ligand.pdb                # Extracted small molecule
│   └── README_STRUCTURES.txt     # Source and prep steps
│
├── parameters/                   # tleap and mdin files
│   ├── leaprc.input              # Master tleap script
│   ├── ligand.mol2               # Antechamber output (GAFF atom types + charges)
│   ├── ligand.frcmod             # parmchk2 output (missing parameters)
│   ├── mdin/
│   │   ├── em.mdin              # Energy minimization
│   │   ├── heat.mdin            # NVT heating
│   │   └── prod.mdin            # NPT production
│   └── mmpbsa.in                # MMPBSA.py settings
│
├── runs/                         # Simulation output
│   ├── 2026_04_test/
│   │   ├── system.prmtop         # Topology (from tleap)
│   │   ├── system.inpcrd         # Initial coordinates
│   │   ├── em/                   # Minimization output
│   │   │   ├── em.mdout
│   │   │   └── em.rst7
│   │   ├── heat/
│   │   │   ├── heat.mdout
│   │   │   └── heat.rst7
│   │   └── prod/
│   │       ├── prod.mdout
│   │       ├── prod.nc           ← main trajectory
│   │       └── prod.rst7
│
├── analysis/
│   ├── cpptraj/
│   │   ├── analyze.cpptraj      # cpptraj batch script
│   │   ├── rmsd.dat             # RMSD output
│   │   ├── rmsf.dat             # Per-residue RMSF
│   │   └── hbonds.dat           # Hydrogen bond data
│   └── mmpbsa/
│       ├── FINAL_MMGBSA_RESULTS.dat
│       └── DELTA_MMGBSA_DECOMP.dat
│
└── scripts/                      # Submission scripts
    ├── submit_sander.sh
    └── submit_mmpbsa.sh
```


---

## APPENDIX C — AmberTools File Types Reference

### Input Files

| Extension | Tool | Purpose |
|-----------|------|---------|
| `.pdb` | tleap, cpptraj | Starting structure and atom coordinates |
| `.mol2` | antechamber → tleap | Small molecule with GAFF atom types + AM1-BCC charges |
| `.frcmod` | parmchk2 → tleap | Supplementary force field parameters for non-standard atoms |
| `.mdin` | sander | Simulation control parameters |
| `.leaprc` / `leaprc.input` | tleap | System preparation workflow |
| `.mmpbsa.in` | MMPBSA.py | Binding free energy parameters |

### Output Files

| Extension | Source Tool | Contains | Keep? |
|-----------|-------------|---------|-------|
| `.prmtop` | tleap | Topology: atom types, bonds, force field | ✅ Always |
| `.inpcrd` | tleap | Initial atomic coordinates | ✅ Always |
| `.rst7` | sander | Restart: coordinates + velocities | ✅ All of them |
| `.mdout` | sander | Energy log: Etot, Etemp, pressure per step | ✅ Always |
| `.nc` or `.mdcrd` | sander | Trajectory: atomic positions over time | ✅ Main output |
| `.dat` | cpptraj | Analysis output (RMSD, RMSF, H-bonds, etc.) | ✅ Results |
| `FINAL_*` | MMPBSA.py | Binding free energy summary | ✅ Results |


---

## APPENDIX D — Storage and Resource Requirements

### Disk Space for CPU Simulations

| System Size | Duration | Trajectory (NetCDF) | Total | Notes |
|---|---|---|---|---|
| 1K atoms | 10 ns | 50 MB | 100 MB | Quick test |
| 5K atoms | 100 ns | 1 GB | 2 GB | Small protein |
| 10K atoms | 100 ns | 2 GB | 5 GB | Medium system |
| 50K atoms | 100 ns | 5 GB | 15 GB | Large system |

### CPU Requirements for sander.MPI

| System Size | Ranks | Expected (ns/day) | Wall Time / 100 ns |
|---|---|---|---|
| < 1K atoms | 1–4 | ~5 ns/day | ~20 hours |
| 1–5K atoms | 4–8 | ~2 ns/day | ~50 hours |
| 5–10K atoms | 8–16 | ~1 ns/day | ~100 hours |
| > 20K atoms | 16–32 | < 0.5 ns/day | Consider GPU |

### MMPBSA.py Resources

| Frames | Method | Cores | Wall Time |
|---|---|---|---|
| 50 frames | MM-GBSA | 4 | ~15 min |
| 200 frames | MM-GBSA | 8 | ~30 min |
| 50 frames | MM-PBSA | 4 | ~1 hour |
| 50 frames | MM-PBSA + nmode | 8 | ~4 hours |


---

## APPENDIX E — Documentation, Tutorials, and Resources

### Official AmberTools Resources

- 📖 **Amber/AmberTools Manual (PDF)**: https://ambermd.org/doc12/Amber25.pdf
  - Chapters 2–4: tleap and force field loading
  - Chapter 5: antechamber / GAFF
  - Chapter 6: sander (mdin format and all parameters)
  - Chapter 12: cpptraj (complete analysis reference)
  - Chapter 34: MMPBSA.py

- 🎓 **Official Tutorials**: https://ambermd.org/tutorials/
  - Tutorial 0: Alanine dipeptide in water (beginner)
  - Tutorial 4b: Small molecule parameterization (antechamber)
  - Tutorial 3: MM-PBSA binding free energy
  - Tutorial 2: Protein-ligand complex

- 🔗 **RCSB PDB** (structures): https://www.rcsb.org/
- 🔗 **AmberHub** (community): https://amberhub.chpc.utah.edu/
- 📧 **Mailing list**: https://lists.ambermd.org/mailman/listinfo/amber

### Local Documentation


In [ ]:
%%bash
module load ambertools 2>/dev/null

echo "=== Local AmberTools Documentation ==="

AMBER_HOME=$(amber.python -c "import os; print(os.environ.get('AMBERHOME','not-set'))" 2>/dev/null || echo "not-set")

if [ "$AMBER_HOME" != "not-set" ] && [ -d "$AMBER_HOME" ]; then
    echo ""
    echo "✅ AmberTools root: $AMBER_HOME"
    ls "$AMBER_HOME/doc/" 2>/dev/null | head -10 || echo "(no doc dir)"
    echo ""
    echo "=== Tutorial examples ==="
    ls "$AMBER_HOME/test/" 2>/dev/null | head -15 || echo "(no test dir)"
else
    echo "⚠️  AMBERHOME not set — try: module load ambertools && echo \$AMBERHOME"
fi


### Key Scientific References

**AmberTools citation:**
D.A. Case et al. "AmberTools." *J. Chem. Inf. Model.* 63, 6183–6191 (2023).

**cpptraj:**
D.R. Roe & T.E. Cheatham. *J. Chem. Theory Comput.* 9, 3084–3095 (2013).

**GAFF2 / antechamber:**
J. Wang et al. *J. Comput. Chem.* 25, 1157–1174 (2004).

**MM-PBSA / MM-GBSA:**
N. Homeyer & H. Gohlke. *Mol. Inform.* 31, 114–122 (2012).

---

This notebook **connects the free AmberTools ecosystem** to production-ready HPC workflows. For GPU acceleration and multi-microsecond production runs, see the companion **Amber Molecular Dynamics Workflow Guide** notebook.
